# 02 Qwen JSONL Formatter

This notebook converts the cleaned medical datasets in `/content/sample_data/cleaned` into production-ready Qwen chat JSONL files.

Outputs:

- `/content/sample_data/qwen_ready/train.jsonl`
- `/content/sample_data/qwen_ready/val.jsonl`
- `/content/sample_data/qwen_ready/test.jsonl`

The pipeline is designed to:

- load cleaned data safely
- detect conversational vs structured schemas
- normalize medical tone
- filter low-quality or unsafe samples
- build medication safety lookups
- create deterministic train/validation/test splits
- export valid Qwen-compatible JSONL

In [ ]:
%%capture
!pip -q install pandas tqdm

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import math
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

pd.options.display.max_colwidth = 160
pd.options.mode.chained_assignment = None

SEED = 42
RNG = random.Random(SEED)
random.seed(SEED)

CLEAN_DIR = Path("/content/sample_data/cleaned")
OUTPUT_DIR = Path("/content/sample_data/qwen_ready")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {
    "GenMedGPT-5k.jsonl": "conversational",
    "HealthCareMagic-100k.jsonl": "conversational",
    "iCliniq.jsonl": "conversational",
    "disease_database_mini.csv": "structured",
    "format_dataset.csv": "structured",
    "indications.tsv": "lookup",
    "side-effects.tsv": "lookup",
    "side-effect-terms.tsv": "lookup",
}

SYSTEM_PROMPT = (
    "You are a careful medical assistant for educational support. "
    "Explain likely possibilities without claiming certainty, use calm natural language, "
    "and focus on practical next steps, common evaluations, and relevant self-care. "
    "When medications are mentioned, keep suggestions conservative and safety-aware, "
    "taking allergies, pregnancy, age, other medicines, and medical history into account. "
    "Do not blindly prescribe dangerous drugs, do not replace a physician, and recommend "
    "urgent care only when red-flag symptoms make it medically appropriate."
)

MIN_USER_WORDS = 4
MIN_ASSISTANT_WORDS = 25
MIN_ICLINIQ_WORDS = 35
MAX_STRUCTURED_SYMPTOMS = 5
MAX_TESTS = 4
MAX_SIDE_EFFECTS = 3
EXPORT_ROLES = ("system", "user", "assistant")
USER_TEXT_KEYS = ("input", "question", "query", "patient_question", "patient_input", "patient", "prompt")
ASSISTANT_TEXT_KEYS = ("output", "answer", "response", "assistant", "doctor_answer", "answer_clean")

LOW_RISK_MEDICATIONS = {
    "acetaminophen",
    "paracetamol",
    "ibuprofen",
    "naproxen",
    "cetirizine",
    "loratadine",
    "famotidine",
    "omeprazole",
    "saline nasal",
    "psyllium",
    "polyethylene glycol",
    "senna",
    "melatonin",
    "sumatriptan",
    "rizatriptan",
}

HIGH_ALERT_PATTERNS = [
    "alprazolam",
    "clonazepam",
    "lorazepam",
    "diazepam",
    "opioid",
    "oxycodone",
    "hydrocodone",
    "morphine",
    "fentanyl",
    "methadone",
    "warfarin",
    "apixaban",
    "rivaroxaban",
    "insulin",
    "rituximab",
    "adalimumab",
    "vinorelbine",
    "cyclophosphamide",
    "chemotherapy",
    "isotretinoin",
]

In [ ]:
def iter_candidate_paths(base_dir: Path, expected_name: str) -> Iterator[Path]:
    expected = base_dir / expected_name
    yield expected

    stem = Path(expected_name).stem
    suffix = Path(expected_name).suffix
    pattern = f"{stem}*{suffix}"

    for folder in (base_dir, base_dir.parent):
        if folder.exists():
            for match in sorted(folder.glob(pattern)):
                yield match


def resolve_dataset_file(base_dir: Path, expected_name: str) -> Path:
    seen = set()
    for candidate in iter_candidate_paths(base_dir, expected_name):
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists() and candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Unable to resolve dataset file: {expected_name}")


def read_first_json_record(path: Path) -> Optional[dict]:
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                return json.loads(line)
    return None


def read_tabular_columns(path: Path, sep: str) -> List[str]:
    frame = pd.read_csv(path, sep=sep, nrows=2, dtype=str)
    return frame.columns.tolist()


def detect_dataset_kind(path: Path) -> Tuple[str, List[str]]:
    if path.suffix.lower() == ".jsonl":
        record = read_first_json_record(path)
        if record is None:
            return "empty_jsonl", []
        columns = sorted(record.keys())
        if {"input", "output"}.issubset(record.keys()):
            return "conversational", columns
        if {"input", "answer_icliniq", "answer_chatgpt"}.intersection(record.keys()):
            return "conversational", columns
        if any(key in record for key in USER_TEXT_KEYS) and any(key in record for key in ASSISTANT_TEXT_KEYS):
            return "conversational", columns
        return "unknown_jsonl", columns

    sep = "\t" if path.suffix.lower() == ".tsv" else ","
    columns = read_tabular_columns(path, sep=sep)
    lowered = {column.lower() for column in columns}

    if {"disease", "symptom"}.issubset(lowered):
        return "structured", columns
    if {"drugbank_name", "side_effect_name"}.issubset(lowered) or {"concept_name", "drugbank_name"}.issubset(lowered):
        return "lookup", columns
    return "unknown_table", columns


def build_manifest(base_dir: Path) -> pd.DataFrame:
    rows = []
    for expected_name, requested_kind in EXPECTED_FILES.items():
        resolved_path = resolve_dataset_file(base_dir, expected_name)
        detected_kind, columns = detect_dataset_kind(resolved_path)
        rows.append(
            {
                "expected_name": expected_name,
                "resolved_name": resolved_path.name,
                "detected_kind": detected_kind,
                "requested_kind": requested_kind,
                "size_bytes": resolved_path.stat().st_size,
                "columns": ", ".join(columns[:12]),
                "path": str(resolved_path),
            }
        )
    return pd.DataFrame(rows).sort_values(["requested_kind", "expected_name"]).reset_index(drop=True)


manifest_df = build_manifest(CLEAN_DIR)
display(manifest_df)

In [ ]:
PROMPT_INJECTION_RE = re.compile(
    r"ignore\s+previous|disregard\s+all\s+instructions|system\s+prompt|developer\s+message|jailbreak|do\s+anything\s+now",
    re.IGNORECASE,
)
META_RE = re.compile(
    r"as an ai|language model|chatgpt|openai|i cannot diagnose|i can't diagnose|virtual assistant|welcome to chat\s*doctor|welcome to chatdoctor|chat\s*doctor|chatdoctor",
    re.IGNORECASE,
)
GARBAGE_RE = re.compile(r"[\uFFFD]|\b(?:lorem ipsum|asdf|qwerty)\b", re.IGNORECASE)
DANGEROUS_RE = re.compile(
    r"bleach|kerosene|mercury|double the dose|stop insulin|skip emergency|no need to seek care|share antibiotics|leftover antibiotics|need prescription antibiotics|start antibiotics|you need antibiotics",
    re.IGNORECASE,
)
DOSING_RE = re.compile(
    r"\b\d+\s*(?:mg|mcg|g|ml)\b|\b(?:once|twice|three times)\s+(?:daily|a day)\b|\bevery\s+\d+\s*(?:hours|hrs?)\b",
    re.IGNORECASE,
)
ROLE_LEAKAGE_RE = re.compile(r"\b(?:patient|doctor|assistant|user)\s*:", re.IGNORECASE)
SEVERE_RE = re.compile(
    r"chest pain|shortness of breath|trouble breathing|one-sided weakness|seizure|fainting|coughing blood|bloody stool|vomiting blood|suicid|severe dehydration|pregnan.*bleeding|anaphyl",
    re.IGNORECASE,
)
ANXIETY_RE = re.compile(r"panic|anxiety|anxious|worry|worried|fear|palpitations", re.IGNORECASE)


def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFKC", str(text or ""))


def normalize_whitespace(text: str) -> str:
    text = normalize_unicode(text)
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def printable_ratio(text: str) -> float:
    if not text:
        return 0.0
    printable = sum(1 for char in text if char.isprintable() or char in "\n\t")
    return printable / max(1, len(text))


def normalize_role_text(text: str) -> str:
    text = normalize_whitespace(text)
    text = re.sub(r"^\s*(hello|hi|dear patient|dear user)[,!: -]+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*(doctor[,\s]+)+", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*welcome to chat\s*doctor\s*forum\.?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*welcome to chatdoctor\.?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bWell, based on what you're telling me,\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\bI would recommend\b", "It would be reasonable to discuss", text, flags=re.IGNORECASE)
    text = re.sub(r"\bYou should take\b", "A clinician may consider", text, flags=re.IGNORECASE)
    text = re.sub(r"\bYou should undergo\b", "Common evaluations may include", text, flags=re.IGNORECASE)
    text = re.sub(r"\bstart taking\b", "A clinician may consider starting", text, flags=re.IGNORECASE)
    text = re.sub(r"\bI think you need\b", "An in-person clinician may decide whether you need", text, flags=re.IGNORECASE)
    text = re.sub(r"\bwe should conduct\b", "Common evaluations may include", text, flags=re.IGNORECASE)
    text = re.sub(r"\bwe need to perform\b", "Common evaluations may include", text, flags=re.IGNORECASE)
    text = re.sub(r"\bwe will need to perform\b", "Common evaluations may include", text, flags=re.IGNORECASE)
    text = re.sub(r"\byou may be suffering from\b", "this could be related to", text, flags=re.IGNORECASE)
    text = re.sub(r"\byou are suffering from\b", "this could be related to", text, flags=re.IGNORECASE)
    text = re.sub(r"\bit seems like you are suffering from\b", "this could be related to", text, flags=re.IGNORECASE)
    text = re.sub(r"\bit sounds like you have\b", "this could fit with", text, flags=re.IGNORECASE)
    text = re.sub(r"\bit sounds like you may have\b", "this could fit with", text, flags=re.IGNORECASE)
    text = re.sub(r"\byou may have\b", "this could be related to", text, flags=re.IGNORECASE)
    text = re.sub(r"\bdefinitely\b", "possibly", text, flags=re.IGNORECASE)
    text = re.sub(r"\bcertainly\b", "likely", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+([?.!,;:])", r"\1", text)
    return text.strip(" -\n\t")


def safe_literal_list(value: object) -> List[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [normalize_whitespace(item) for item in value if normalize_whitespace(item)]

    text = normalize_whitespace(str(value))
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [normalize_whitespace(item) for item in parsed if normalize_whitespace(item)]
    except Exception:
        pass

    pieces = [piece.strip(" []'\"") for piece in text.split(",")]
    return [normalize_whitespace(piece) for piece in pieces if normalize_whitespace(piece)]


def natural_join(items: List[str]) -> str:
    items = [normalize_whitespace(item) for item in items if normalize_whitespace(item)]
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f"{items[0]} and {items[1]}"
    return ", ".join(items[:-1]) + f", and {items[-1]}"


def sentence_case(text: str) -> str:
    text = normalize_whitespace(text)
    if not text:
        return ""
    return text[0].upper() + text[1:]


def hash_text(*parts: str) -> str:
    payload = "||".join(normalize_whitespace(part).lower() for part in parts)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()


def quality_reject_reason(user_text: str, assistant_text: str, min_answer_words: int = MIN_ASSISTANT_WORDS) -> Optional[str]:
    combined = f"{user_text}\n{assistant_text}"
    if printable_ratio(combined) < 0.95 or GARBAGE_RE.search(combined):
        return "garbage_or_encoding"
    if PROMPT_INJECTION_RE.search(combined):
        return "prompt_injection"
    if DANGEROUS_RE.search(assistant_text):
        return "dangerous_advice"
    if DOSING_RE.search(assistant_text):
        return "dosing_instruction"
    if ROLE_LEAKAGE_RE.search(assistant_text):
        return "role_leakage"
    if META_RE.search(assistant_text):
        return "meta_or_branding"
    if len(normalize_whitespace(user_text).split()) < MIN_USER_WORDS:
        return "short_user"
    if len(normalize_whitespace(assistant_text).split()) < min_answer_words:
        return "short_answer"
    return None


def trim_to_sentences(text: str, max_sentences: int = 2) -> str:
    text = normalize_role_text(text)
    if not text:
        return ""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return " ".join(sentences[:max_sentences]).strip()


def build_red_flag_guidance(text: str) -> str:
    lowered = normalize_whitespace(text).lower()
    if re.search(r"suicid|self-harm", lowered):
        return "Urgent mental health support is appropriate if there are thoughts of self-harm, inability to stay safe, or severe worsening."
    if re.search(r"pregnan.*bleeding", lowered):
        return "Prompt in-person care is appropriate for pregnancy with bleeding, severe pain, fainting, or significant dizziness."
    if SEVERE_RE.search(lowered):
        return "Urgent evaluation is appropriate if symptoms include worsening chest pain, trouble breathing, fainting, one-sided weakness, severe dehydration, or bleeding."
    return ""


def classify_primary_topic(user_text: str, assistant_text: str) -> str:
    combined = f"{user_text} {assistant_text}"
    if SEVERE_RE.search(combined):
        return "emergency"
    if ANXIETY_RE.search(combined):
        return "anxiety"
    if assistant_text.lower().count("urgent") + assistant_text.lower().count("emergency") >= 2:
        return "disclaimer_heavy"
    return "general"


def make_chat_sample(source: str, user_text: str, assistant_text: str, sample_id: str) -> dict:
    topic = classify_primary_topic(user_text, assistant_text)
    return {
        "source": source,
        "sample_id": sample_id,
        "primary_topic": topic,
        "hash": hash_text(user_text, assistant_text),
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": normalize_role_text(user_text)},
            {"role": "assistant", "content": normalize_role_text(assistant_text)},
        ],
    }

In [ ]:
def canonical_medication_name(name: str) -> str:
    name = normalize_whitespace(name).lower()
    name = re.sub(r"\([^)]*\)", "", name)
    name = re.sub(r"[^a-z0-9+\-/ ]+", " ", name)
    name = re.sub(r"\s+", " ", name)
    return name.strip()


def build_medication_lookups(clean_dir: Path) -> dict:
    indications_path = resolve_dataset_file(clean_dir, "indications.tsv")
    side_effects_path = resolve_dataset_file(clean_dir, "side-effects.tsv")
    side_effect_terms_path = resolve_dataset_file(clean_dir, "side-effect-terms.tsv")

    medication_to_indications = defaultdict(set)
    condition_to_medications = defaultdict(set)
    medication_to_side_effects = defaultdict(set)
    side_effect_terms = set()

    for chunk in tqdm(
        pd.read_csv(indications_path, sep="\t", dtype=str, chunksize=100_000),
        desc="Loading indications",
    ):
        chunk = chunk.fillna("")
        for row in chunk.itertuples(index=False):
            drug_name = canonical_medication_name(getattr(row, "drugbank_name", ""))
            concept_name = normalize_whitespace(getattr(row, "concept_name", ""))
            if not drug_name or not concept_name:
                continue
            medication_to_indications[drug_name].add(concept_name)
            condition_to_medications[concept_name.lower()].add(drug_name)

    for chunk in tqdm(
        pd.read_csv(side_effects_path, sep="\t", dtype=str, chunksize=100_000),
        desc="Loading side effects",
    ):
        chunk = chunk.fillna("")
        for row in chunk.itertuples(index=False):
            drug_name = canonical_medication_name(getattr(row, "drugbank_name", ""))
            effect_name = normalize_whitespace(getattr(row, "side_effect_name", ""))
            if drug_name and effect_name:
                medication_to_side_effects[drug_name].add(effect_name)

    for chunk in tqdm(
        pd.read_csv(side_effect_terms_path, sep="\t", dtype=str, chunksize=100_000),
        desc="Loading side effect terms",
    ):
        chunk = chunk.fillna("")
        side_effect_terms.update(
            normalize_whitespace(value)
            for value in chunk["side_effect_name"].tolist()
            if normalize_whitespace(value)
        )

    return {
        "medication_to_indications": medication_to_indications,
        "condition_to_medications": condition_to_medications,
        "medication_to_side_effects": medication_to_side_effects,
        "side_effect_terms": side_effect_terms,
    }


def medication_is_high_alert(name: str) -> bool:
    canonical = canonical_medication_name(name)
    return any(pattern in canonical for pattern in HIGH_ALERT_PATTERNS)


def indication_match_score(disease: str, medication_name: str, lookups: dict) -> int:
    disease_terms = set(re.findall(r"[a-z]+", disease.lower()))
    indications = lookups["medication_to_indications"].get(canonical_medication_name(medication_name), set())
    best = 0
    for indication in indications:
        indication_terms = set(re.findall(r"[a-z]+", indication.lower()))
        overlap = len(disease_terms & indication_terms)
        best = max(best, overlap)
    return best


def select_vetted_medications(disease: str, medications: List[str], lookups: dict, max_items: int = 2) -> List[str]:
    candidates = []
    for medication in medications:
        canonical = canonical_medication_name(medication)
        if not canonical:
            continue
        if medication_is_high_alert(canonical):
            continue

        score = indication_match_score(disease, canonical, lookups)
        low_risk_match = any(token in canonical for token in LOW_RISK_MEDICATIONS)

        if low_risk_match:
            score += 3

        if score <= 0:
            continue

        candidates.append((score, sentence_case(medication)))

    candidates.sort(key=lambda item: (-item[0], item[1].lower()))
    deduped = []
    seen = set()
    for _, medication in candidates:
        key = medication.lower()
        if key not in seen:
            seen.add(key)
            deduped.append(medication)
        if len(deduped) >= max_items:
            break
    return deduped


LOOKUPS = build_medication_lookups(CLEAN_DIR)
print("Medication indications:", len(LOOKUPS["medication_to_indications"]))
print("Medication side-effect entries:", len(LOOKUPS["medication_to_side_effects"]))
print("Known side-effect terms:", len(LOOKUPS["side_effect_terms"]))

In [ ]:
def choose_icliniq_answer(record: dict) -> Tuple[Optional[str], str]:
    preferred = normalize_role_text(record.get("answer_icliniq", ""))
    fallback = normalize_role_text(record.get("answer_chatgpt", ""))

    if preferred:
        chatdoctor_mentions = len(re.findall(r"chat\s*doctor", preferred, flags=re.IGNORECASE))
        if chatdoctor_mentions > 1:
            preferred = ""
        elif chatdoctor_mentions == 1:
            preferred = re.sub(r"chat\s*doctor", "", preferred, flags=re.IGNORECASE).strip()

    reason = quality_reject_reason(
        normalize_role_text(record.get("input", "")),
        preferred,
        min_answer_words=MIN_ICLINIQ_WORDS,
    ) if preferred else "empty_preferred"

    if preferred and reason is None:
        return preferred, "answer_icliniq"

    fallback_reason = quality_reject_reason(
        normalize_role_text(record.get("input", "")),
        fallback,
        min_answer_words=MIN_ICLINIQ_WORDS,
    ) if fallback else "empty_fallback"

    if fallback and fallback_reason is None:
        return fallback, "answer_chatgpt"

    return None, fallback_reason if fallback_reason != "empty_fallback" else reason


def file_has_content(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                return True
    return False


def first_present_text(record: dict, keys: Tuple[str, ...]) -> str:
    for key in keys:
        value = normalize_role_text(record.get(key, ""))
        if value:
            return value
    return ""


def choose_generic_answer(record: dict) -> Tuple[Optional[str], str]:
    assistant_text = first_present_text(record, ASSISTANT_TEXT_KEYS)
    if not assistant_text:
        return None, "empty_answer"
    return assistant_text, "generic_answer"


def process_generic_conversational_dataset(
    path: Path,
    stats: dict,
    answer_selector,
    min_answer_words: int = MIN_ASSISTANT_WORDS,
) -> List[dict]:
    if not file_has_content(path):
        stats["skipped"][f"{path.name}:empty_file"] += 1
        return []

    samples = []
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(tqdm(handle, desc=f"Processing {path.name}")):
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                stats["rejected"][f"{path.name}:malformed_json"] += 1
                continue

            user_text = first_present_text(record, USER_TEXT_KEYS)
            assistant_text, answer_source = answer_selector(record)
            if not user_text:
                stats["rejected"][f"{path.name}:missing_user"] += 1
                continue
            if not assistant_text:
                stats["rejected"][f"{path.name}:{answer_source}"] += 1
                continue

            reason = quality_reject_reason(user_text, assistant_text, min_answer_words=min_answer_words)
            if reason is not None:
                stats["rejected"][f"{path.name}:{reason}"] += 1
                continue

            sample = make_chat_sample(path.name, user_text, assistant_text, f"{path.stem}-{index}")
            if answer_source:
                sample["answer_source"] = answer_source
            samples.append(sample)
            stats["accepted"][path.name] += 1
    return samples


def process_genmed_dataset(path: Path, stats: dict) -> List[dict]:
    return process_generic_conversational_dataset(
        path=path,
        stats=stats,
        answer_selector=lambda record: (normalize_role_text(record.get("output", "")), "output"),
        min_answer_words=MIN_ASSISTANT_WORDS,
    )


def process_icliniq_dataset(path: Path, stats: dict) -> List[dict]:
    samples = process_generic_conversational_dataset(
        path=path,
        stats=stats,
        answer_selector=choose_icliniq_answer,
        min_answer_words=MIN_ICLINIQ_WORDS,
    )
    for sample in samples:
        stats["iCliniq_answer_choice"][sample.get("answer_source", "unknown")] += 1
    return samples


def process_healthcaremagic_dataset(path: Path, stats: dict) -> List[dict]:
    return process_generic_conversational_dataset(
        path=path,
        stats=stats,
        answer_selector=choose_generic_answer,
        min_answer_words=MIN_ASSISTANT_WORDS,
    )

In [ ]:
USER_TEMPLATES = [
    "I have {symptoms}. What could be going on?",
    "I've been dealing with {symptoms}. What are some possible explanations and what is usually checked?",
    "I keep noticing {symptoms}. What condition could fit this pattern?",
    "For the last few days I've had {symptoms}. What are the common next steps?",
]

LEAD_TEMPLATES = [
    "This symptom pattern can be seen with {disease}.",
    "{disease} is one possible explanation for symptoms like these.",
    "One condition that can fit this combination of symptoms is {disease}.",
    "These symptoms may be associated with {disease}.",
]

FOLLOWUP_TEMPLATES = [
    "If symptoms are new, persistent, or getting worse, a clinician can help narrow the diagnosis.",
    "If the pattern keeps recurring or becomes more intense, an in-person evaluation can clarify the cause.",
    "If symptoms continue or do not improve, follow-up care can help confirm the diagnosis and guide treatment.",
    "",
]


def stable_rng(*parts: str) -> random.Random:
    seed = int(hash_text(*parts)[:12], 16)
    return random.Random(seed)


def clean_reason_text(reason: object) -> str:
    text = normalize_role_text(reason or "")
    if not text:
        return ""
    if len(set(re.findall(r"[a-z]+", text.lower()))) < 6:
        return ""
    if "," in text and "." not in text and len(text.split()) < 12:
        return ""
    trimmed = trim_to_sentences(text, max_sentences=2)
    return trimmed if len(trimmed.split()) >= 10 else ""


def build_structured_user_text(symptoms: List[str], disease: str, row_key: str) -> str:
    rng = stable_rng(row_key, disease)
    symptom_slice = symptoms[:MAX_STRUCTURED_SYMPTOMS]
    if len(symptom_slice) >= 4:
        start = rng.randint(0, max(0, len(symptom_slice) - 4))
        symptom_slice = symptom_slice[start:start + 4]
    symptom_text = natural_join([sentence_case(symptom.lower()) for symptom in symptom_slice])
    template = rng.choice(USER_TEMPLATES)
    return template.format(symptoms=symptom_text.lower())


def build_structured_assistant_text(
    disease: str,
    symptoms: List[str],
    tests: List[str],
    medications: List[str],
    reason: str,
    row_key: str,
    lookups: dict,
) -> str:
    rng = stable_rng(row_key, disease, reason)
    parts = [rng.choice(LEAD_TEMPLATES).format(disease=disease)]

    if reason:
        parts.append(reason)

    if tests:
        parts.append(f"Common evaluations may include {natural_join(tests[:MAX_TESTS])}.")

    red_flag_guidance = build_red_flag_guidance(" ".join([disease] + symptoms))
    vetted_medications = [] if red_flag_guidance else select_vetted_medications(disease, medications, lookups, max_items=2)

    if vetted_medications:
        parts.append(
            f"Depending on the confirmed diagnosis and your medical history, clinicians sometimes use {natural_join(vetted_medications)}."
        )

        side_effects = []
        for medication in vetted_medications:
            canonical = canonical_medication_name(medication)
            side_effects.extend(sorted(lookups["medication_to_side_effects"].get(canonical, set()))[:2])
        side_effects = [effect for effect in side_effects if effect][:MAX_SIDE_EFFECTS]

        if side_effects:
            parts.append(f"If medication is prescribed, side effects can include {natural_join(side_effects)}.")
    else:
        parts.append(
            "Treatment depends on the confirmed diagnosis and may involve supportive care or clinician-guided prescription treatment rather than starting a new medicine on your own."
        )

    if red_flag_guidance:
        parts.append(red_flag_guidance)
    else:
        followup = rng.choice(FOLLOWUP_TEMPLATES)
        if followup:
            parts.append(followup)

    return normalize_role_text(" ".join(part for part in parts if part))


def process_structured_dataset(path: Path, stats: dict, lookups: dict) -> List[dict]:
    samples = []
    sep = "\t" if path.suffix.lower() == ".tsv" else ","
    for chunk in tqdm(
        pd.read_csv(path, sep=sep, dtype=str, chunksize=10_000),
        desc=f"Processing {path.name}",
    ):
        chunk = chunk.fillna("")
        for row in chunk.to_dict(orient="records"):
            disease = sentence_case(row.get("disease", ""))
            symptoms = safe_literal_list(row.get("symptom", ""))
            reason = clean_reason_text(row.get("reason", ""))

            tests = safe_literal_list(row.get("medical_tests", "")) or safe_literal_list(row.get("testsandprocedures", ""))
            medications = safe_literal_list(row.get("medications", "")) or safe_literal_list(row.get("commonmedications", ""))

            if not disease or len(symptoms) < 2:
                stats["rejected"][f"{path.name}:missing_core_fields"] += 1
                continue

            row_key = f"{path.stem}-{row.get('idx', len(samples))}"
            user_text = build_structured_user_text(symptoms, disease, row_key)
            assistant_text = build_structured_assistant_text(disease, symptoms, tests, medications, reason, row_key, lookups)

            reject_reason = quality_reject_reason(user_text, assistant_text, min_answer_words=MIN_ASSISTANT_WORDS)
            if reject_reason is not None:
                stats["rejected"][f"{path.name}:{reject_reason}"] += 1
                continue

            sample = make_chat_sample(path.name, user_text, assistant_text, row_key)
            samples.append(sample)
            stats["accepted"][path.name] += 1
    return samples

In [ ]:
def deduplicate_samples(samples: List[dict], stats: dict) -> List[dict]:
    deduped = []
    seen = set()
    for sample in samples:
        if sample["hash"] in seen:
            stats["duplicates_removed"] += 1
            continue
        seen.add(sample["hash"])
        deduped.append(sample)
    return deduped


def balance_samples(samples: List[dict], stats: dict) -> List[dict]:
    caps = {
        "anxiety": 0.18,
        "emergency": 0.10,
        "disclaimer_heavy": 0.06,
    }
    by_topic = defaultdict(list)
    for sample in samples:
        by_topic[sample["primary_topic"]].append(sample)

    keep_hashes = set(sample["hash"] for sample in samples)
    total = len(samples)

    for topic, cap in caps.items():
        topic_samples = by_topic.get(topic, [])
        max_allowed = max(50, int(total * cap))
        if len(topic_samples) <= max_allowed:
            continue

        rng = stable_rng(topic, str(total))
        ordered = sorted(topic_samples, key=lambda item: item["hash"])
        rng.shuffle(ordered)
        to_drop = ordered[max_allowed:]

        for sample in to_drop:
            keep_hashes.discard(sample["hash"])
        stats["balanced_removed"][topic] += len(to_drop)

    balanced = [sample for sample in samples if sample["hash"] in keep_hashes]
    return balanced


def split_counts(n_items: int, train_ratio: float = 0.90, val_ratio: float = 0.05) -> Tuple[int, int, int]:
    if n_items <= 2:
        return n_items, 0, 0

    train_count = int(n_items * train_ratio)
    val_count = int(n_items * val_ratio)
    test_count = n_items - train_count - val_count

    if n_items >= 20:
        val_count = max(1, val_count)
        test_count = max(1, test_count)

    while train_count + val_count + test_count > n_items:
        if train_count >= val_count and train_count >= test_count and train_count > 1:
            train_count -= 1
        elif val_count >= test_count and val_count > 0:
            val_count -= 1
        else:
            test_count -= 1

    while train_count + val_count + test_count < n_items:
        train_count += 1

    return train_count, val_count, test_count


def stratified_split(samples: List[dict]) -> Dict[str, List[dict]]:
    grouped = defaultdict(list)
    for sample in samples:
        grouped[sample["source"]].append(sample)

    split_map = {"train": [], "val": [], "test": []}
    for source, source_samples in grouped.items():
        rng = stable_rng(source, str(len(source_samples)))
        ordered = list(sorted(source_samples, key=lambda item: item["hash"]))
        rng.shuffle(ordered)

        train_count, val_count, test_count = split_counts(len(ordered))
        split_map["train"].extend(ordered[:train_count])
        split_map["val"].extend(ordered[train_count:train_count + val_count])
        split_map["test"].extend(ordered[train_count + val_count:train_count + val_count + test_count])

    for split_name in split_map:
        rng = stable_rng(split_name, str(len(split_map[split_name])))
        ordered = list(sorted(split_map[split_name], key=lambda item: item["hash"]))
        rng.shuffle(ordered)
        split_map[split_name] = ordered

    return split_map


def validate_messages(record: dict) -> None:
    if "messages" not in record or not isinstance(record["messages"], list):
        raise ValueError("Record must contain a messages list.")
    roles = [message.get("role") for message in record["messages"]]
    if tuple(roles) != EXPORT_ROLES:
        raise ValueError(f"Unexpected role order: {roles}")
    for message in record["messages"]:
        content = message.get("content", "")
        if not isinstance(content, str) or not normalize_whitespace(content):
            raise ValueError("Each message must contain non-empty string content.")


def export_jsonl(path: Path, samples: List[dict]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for sample in samples:
            record = {"messages": sample["messages"]}
            validate_messages(record)
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


stats = {
    "accepted": Counter(),
    "rejected": Counter(),
    "skipped": Counter(),
    "duplicates_removed": 0,
    "balanced_removed": Counter(),
    "iCliniq_answer_choice": Counter(),
}

resolved_paths = {name: resolve_dataset_file(CLEAN_DIR, name) for name in EXPECTED_FILES}

# KEEP ONLY conversational datasets for SFT training
ALLOWED_SFT_DATASETS = {
    "GenMedGPT-5k.jsonl",
    "iCliniq.jsonl",
    "HealthCareMagic-100k.jsonl",
}

def detect_structured_content(text: str) -> bool:
    if not text:
        return False
    t = normalize_whitespace(text)
    # explicit delimiters and table markers
    if '|' in t or '\t' in t:
        return True
    # short CSV/TSV-like rows with multiple commas and short tokens
    if t.count(',') >= 2 and len(t.split()) < 30:
        parts = [p.strip() for p in t.split(',') if p.strip()]
        short_parts = sum(1 for p in parts if len(p.split()) <= 4)
        if short_parts >= max(2, len(parts) // 2):
            return True
    # key:value dumps commonly found in tables or exports (drug: aspirin)
    if re.search(r'\b(?:drug|drug_name|drugbank_name|side[_-]?effect|dosage|dose|symptom|test|procedure)\b', t, re.I) and (':' in t or '|' in t or '\t' in t):
        return True
    # lines like 'aspirin | nausea | dizziness' or 'aspirin, nausea, dizziness'
    if re.search(r'^\s*[\w\- ]+(\s*[|,]\s*[\w\- ]+){1,}\s*$', t):
        return True
    return False

def sanity_check_sample(sample: dict, stats: dict) -> bool:
    if not isinstance(sample, dict):
        stats['rejected']['not_dict'] += 1
        return False
    messages = sample.get('messages')
    if not isinstance(messages, list) or len(messages) < 2:
        stats['rejected']['missing_or_invalid_messages'] += 1
        return False
    roles = [m.get('role') for m in messages if isinstance(m, dict)]
    if 'user' not in roles or 'assistant' not in roles:
        stats['rejected']['missing_user_or_assistant'] += 1
        return False
    # collect user and assistant text
    user_text = ' '.join(m.get('content','') for m in messages if m.get('role') == 'user')
    assistant_text = ' '.join(m.get('content','') for m in messages if m.get('role') == 'assistant')
    if not user_text.strip():
        stats['rejected']['empty_user'] += 1
        return False
    if not assistant_text.strip():
        stats['rejected']['empty_assistant'] += 1
        return False
    # reject structured-like content
    if detect_structured_content(user_text) or detect_structured_content(assistant_text):
        stats['rejected']['structured_like'] += 1
        return False
    # reject obvious key:value table rows
    if re.search(r'\b(?:drug:|side_effect:|side effect:|dosage:|dose:)\b', user_text, re.I) or re.search(r'\b(?:drug:|side_effect:|side effect:|dosage:|dose:)\b', assistant_text, re.I):
        stats['rejected']['structured_keys'] += 1
        return False
    return True

# Load only allowed conversational datasets and process them
all_samples = []
for name in ALLOWED_SFT_DATASETS:
    path = resolved_paths.get(name)
    if path is None:
        # tolerate .json vs .jsonl filename variants
        alt = name.replace('.jsonl', '.json') if name.endswith('.jsonl') else name + '.jsonl'
        path = resolved_paths.get(alt)
    if path is None:
        stats['skipped'][f'{name}:missing'] += 1
        continue
    if name.startswith('GenMedGPT'):
        all_samples.extend(process_genmed_dataset(path, stats))
    elif name.startswith('HealthCareMagic'):
        all_samples.extend(process_healthcaremagic_dataset(path, stats))
    elif name.startswith('iCliniq'):
        all_samples.extend(process_icliniq_dataset(path, stats))
    else:
        stats['skipped'][f'{name}:unsupported_type'] += 1

# Pre-filter samples with sanity checks (removes structured/table-like records)
pre_filtered = []
raw_source_counts = Counter(sample.get('source') for sample in all_samples)
for sample in all_samples:
    if sanity_check_sample(sample, stats):
        pre_filtered.append(sample)

pre_filtered_counts = Counter(sample.get('source') for sample in pre_filtered)

total_before_dedup = len(pre_filtered)
all_samples = deduplicate_samples(pre_filtered, stats)
total_after_dedup = len(all_samples)
all_samples = balance_samples(all_samples, stats)
total_after_balance = len(all_samples)

split_map = stratified_split(all_samples)

export_jsonl(OUTPUT_DIR / 'train.jsonl', split_map['train'])
export_jsonl(OUTPUT_DIR / 'val.jsonl', split_map['val'])
export_jsonl(OUTPUT_DIR / 'test.jsonl', split_map['test'])

def sample_token_length(sample: dict) -> int:
    user_text = ' '.join(m.get('content','') for m in sample.get('messages',[]) if m.get('role') == 'user')
    assistant_text = ' '.join(m.get('content','') for m in sample.get('messages',[]) if m.get('role') == 'assistant')
    return len((user_text + ' ' + assistant_text).split())

token_lengths = [sample_token_length(s) for s in all_samples]
avg_token_length = round(sum(token_lengths) / max(1, len(token_lengths)), 2)
max_token_length = max(token_lengths) if token_lengths else 0

final_source_counts = Counter(sample.get('source') for sample in all_samples)
split_counts_report = {split_name: len(records) for split_name, records in split_map.items()}

report = {
    'raw_source_counts': dict(sorted(raw_source_counts.items())),
    'pre_filtered_counts': dict(sorted(pre_filtered_counts.items())),
    'final_source_counts': dict(sorted(final_source_counts.items())),
    'total_before_dedup': total_before_dedup,
    'total_after_dedup': total_after_dedup,
    'total_after_balance': total_after_balance,
    'split_sizes': split_counts_report,
    'avg_token_length': avg_token_length,
    'max_token_length': max_token_length,
    'accepted_per_source': dict(sorted(stats['accepted'].items())),
    'rejected_counts': dict(sorted(stats['rejected'].items())),
    'skipped_counts': dict(sorted(stats['skipped'].items())),
    'duplicates_removed': stats['duplicates_removed'],
    'balanced_removed': dict(sorted(stats['balanced_removed'].items())),
    'icliniq_answer_choice': dict(sorted(stats['iCliniq_answer_choice'].items())),
}

with (OUTPUT_DIR / 'dataset_stats.json').open('w', encoding='utf-8') as handle:
    json.dump(report, handle, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
def validate_export(path: Path, preview_rows: int = 2) -> pd.DataFrame:
    preview = []
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            record = json.loads(line)
            validate_messages(record)
            if index < preview_rows:
                preview.append(
                    {
                        "split": path.stem,
                        "system": record["messages"][0]["content"][:120],
                        "user": record["messages"][1]["content"][:140],
                        "assistant": record["messages"][2]["content"][:180],
                    }
                )
    return pd.DataFrame(preview)


previews = pd.concat(
    [
        validate_export(OUTPUT_DIR / "train.jsonl"),
        validate_export(OUTPUT_DIR / "val.jsonl"),
        validate_export(OUTPUT_DIR / "test.jsonl"),
    ],
    ignore_index=True,
)

display(previews)
print("Exports written to:", OUTPUT_DIR)
print("Files:", sorted(path.name for path in OUTPUT_DIR.glob("*.jsonl")))